## actualizar retiro telef 

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

server_sql = server_zeus
db_sql = "THOTH"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_thoth = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [25]:
query = f"""
	select * from THOTH.dbo.Tmp_LLamadas_Alfin
"""
df_llamadas = pd.read_sql(query, engine_thoth)


print(df_llamadas.columns.tolist())


['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin', 'lead_id', 'Estado_', 'Sub_Estado_', 'Descripcion_', 'Pesos', 'Enlace', 'Fecha_Llam', 'Hora_Llamada', 'RH', 'COD_BCO']


In [6]:
query = f"""
	select * from Alice.prospectos_correos_alfin
"""
df_correo = pd.read_sql(query, engine_mysql)

# set_correo = set(
#     df_formato['agencia_atencion']
#     .dropna()
#     .drop_duplicates()
# )

# set_agencia = set(
#     df_agencia['agencia_correo']
#     .dropna()
#     .drop_duplicates()
# )
# # print(set_correo & set_agencia)
# print(set_agencia - set_correo)
# print(set_correo -set_agencia )

In [11]:
query = f"""
	select * from Alice.prospectos_envio_alfin
"""
df_formulario = pd.read_sql(query, engine_mysql)

In [15]:
df_validacion=df_formulario[df_formulario['estado']=='PROCESADO']

In [10]:
df_gestion_venta=df_correo[df_correo['estado']=='ENVIADO']

In [ ]:
df_gestion_venta

In [20]:
display(df_gestion_venta.head(2))
display(df_validacion.head(2))


,id,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,fecha_envio,intentos_realizados,estado,tipo_carga,fecha_registro,fecha_dia
0,1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,VERDE OSCURO,4000.0,963517955,TRUJILLO CENTRO,2026-07-04,0 days 02:06:01,2026-07-03 11:09:44,2,ENVIADO,AUTOMATICO,2026-07-03 02:06:01,2026-07-03
1,2,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONORE,19998396,ANGEL LUIS PONCE HUARINGA,AMARILLO OSCURO,14000.0,937474638,HUANCAYO,2026-07-04,0 days 02:06:01,2026-07-03 11:14:54,1,ENVIADO,AUTOMATICO,2026-07-03 02:06:01,2026-07-03


,id,hash_duplicado,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion,estado,codigo_http_ms,respuesta_ms,fecha_creacion,fecha_dia
0,3,3a6b361a91aa763c094d1f5c7f164cc2,00000001,TARGET,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,963517955,734265 - TRUJ CENTRO,2026-07-04,4000,Derivacion,PROCESADO,201.0,Registrado en Forms Alfin.,2026-07-03 02:06:01,2026-07-03 10:57:08
1,4,a3faa0a4c0f17ed48834205c8f9db5d0,00000001,TARGET,19998396,ANGEL LUIS PONCE HUARINGA,937474638,734280 - PC HUANCAYO,2026-07-04,14000,Derivacion,PROCESADO,201.0,Registrado en Forms Alfin.,2026-07-03 02:06:01,2026-07-03 10:57:10


In [17]:
df_validacion = df_validacion.rename(
    columns={'fecha_envio': 'fecha_dia'}
)

In [21]:
df_gestion_venta['fecha_dia'] = pd.to_datetime(
    df_gestion_venta['fecha_dia']
).dt.date

df_validacion['fecha_dia'] = pd.to_datetime(
    df_validacion['fecha_dia']
).dt.date

C:\Users\DATA\AppData\Local\Temp\ipykernel_11436\2233122657.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gestion_venta['fecha_dia'] = pd.to_datetime(


In [22]:
df_gestion_venta = df_gestion_venta.merge(
    df_validacion[['dni_cliente', 'fecha_dia']].drop_duplicates(),
    on=['dni_cliente', 'fecha_dia'],
    how='inner'
)

In [ ]:
df_gestion_venta.head()

In [47]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

display(df_llamadas[df_llamadas['Nombre_Campana']=='BOT_ALFIN'].head(10))
# display(df_gestion_venta.head(2))

,Dni,Numero_Campana,Nombre_Campana,DNI_Ejecutivo,Ejecutivo,Fecha_Hora_Llamada,segundos,Fecha_Llamada,Trama_Hora,Estados,Sub_estado,Descripcion,list_description,list_name,PHONE_NUMBER,Fecha_Agenda,Comentarios,Codigo_Paleta,Inicio,Fin,lead_id,Estado_,Sub_Estado_,Descripcion_,Pesos,Enlace,Fecha_Llam,Hora_Llamada,RH,COD_BCO
0,22415967,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 16:04:34,0.0,2026-07-03,16,,,None,Prueba_bot_alfin,Prueba_bot_alfin,995999232,None,None,2,2026-07-03 16:04:34,2026-07-03 16:07:01,20807824.0,None,None,None,None,22415967 995999232,2026-07-03,16,3,None
150,02875483,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:42:33,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,998149248,None,None,2,2026-07-03 11:42:33,2026-07-03 11:44:01,20808332.0,None,None,None,None,02875483 998149248,2026-07-03,11,1,None
151,06679750,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:42:33,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,959198567,None,None,2,2026-07-03 11:42:33,2026-07-03 11:44:01,20802234.0,None,None,None,None,06679750 959198567,2026-07-03,11,1,None
152,10304508,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:43:56,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,995766803,None,None,2,2026-07-03 11:43:56,2026-07-03 11:46:01,20807782.0,None,None,None,None,10304508 995766803,2026-07-03,11,1,None
153,09889201,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:45:53,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,933365706,None,None,2,2026-07-03 11:45:53,2026-07-03 11:48:02,20798503.0,None,None,None,None,09889201 933365706,2026-07-03,11,1,None
154,74587681,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:47:50,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,923350890,None,None,2,2026-07-03 11:47:50,2026-07-03 11:49:01,20798042.0,None,None,None,None,74587681 923350890,2026-07-03,11,1,None
155,47561592,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:48:24,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,940022958,None,None,2,2026-07-03 11:48:24,2026-07-03 11:50:01,20799002.0,None,None,None,None,47561592 940022958,2026-07-03,11,1,None
156,46088450,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:48:53,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,992565214,None,None,2,2026-07-03 11:48:53,2026-07-03 11:50:01,20807129.0,None,None,None,None,46088450 992565214,2026-07-03,11,1,None
157,15655114,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:49:39,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,985382347,None,None,2,2026-07-03 11:49:39,2026-07-03 11:51:02,20805767.0,None,None,None,None,15655114 985382347,2026-07-03,11,1,None
158,32892436,401,BOT_ALFIN,49587612,Bot Bco Alfin,2026-07-03 11:50:08,0.0,2026-07-03,11,,,None,Prueba_bot_alfin,Prueba_bot_alfin,936990203,None,None,2,2026-07-03 11:50:08,2026-07-03 11:52:01,20798762.0,None,None,None,None,32892436 936990203,2026-07-03,11,1,None


In [28]:
df_gestion_venta=df_gestion_venta[['dni_cliente','fecha_dia']]

In [29]:
df_gestion_venta = (
    df_gestion_venta[['dni_cliente', 'fecha_dia']]
    .drop_duplicates()
)

In [30]:
df_gestion_venta = df_gestion_venta.rename(
    columns={
        'dni_cliente': 'Dni',
        'fecha_dia': 'Fecha_Llam'
    }
)

In [31]:
df_llamadas = df_llamadas.merge(
    df_gestion_venta[['Dni', 'Fecha_Llam']].drop_duplicates(),
    on=['Dni', 'Fecha_Llam'],
    how='inner'
)

In [ ]:
df_llamadas = (
    df_llamadas
    .sort_values('Fecha_Llam', ascending=True)
    .drop_duplicates(subset=['Dni', 'Fecha_Llam'], keep='first')
)

In [35]:
df_llamadas=df_llamadas.drop_duplicates(subset=['Dni'])

In [37]:
df_validacion = (
    df_llamadas
    .groupby('Fecha_Llam')['Dni']
    .count()
    .reset_index(name='cantidad_dni')
)

In [39]:
df_validacion.head(10)

,Fecha_Llam,cantidad_dni
0,2026-07-03,235
1,2026-07-06,694
2,2026-07-08,131
3,2026-07-09,24
4,2026-07-10,11
5,2026-07-11,56


In [40]:

ruta_archivo = os.path.join(ruta_csv, 'alfin_ventas.csv')
df_llamadas.to_csv(ruta_archivo, index=False,sep=';')

In [42]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [43]:
df_car=cargar_archivo_csv(spark,'alfin_ventas.csv',';',True)

In [44]:
append_table_SQL(spark,df_car,'ver_ventas_alfin',server_zeus,user_zeus,pwd_zeus,'ODIN')


In [3]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_formato = df_formato.merge(
    df_maestra,
    on='dni_cliente',
    how='left'
)

df_formato['fecha_visita']='2026-07-11'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [4]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [5]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [6]:
df_formato.shape

(1051, 19)

In [20]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [7]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

set()
set()


In [23]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'ICA'}
set()


#### validar el codigo de agencia 

In [24]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'730879 - PAITA', '739580 - ICA'}
set()


In [11]:

df_correo[df_correo['dni_cliente']=='09485785'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
1127,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09485785,LUIS ENRIQUE TORRES BUITRON,NaN,25000,945767367,SAN JUAN DE MIRAFLORES,2026-07-11,13:00:00


In [ ]:
df_correo['tipo_carga']='MANUAL'
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,32926030,CARLOS EDGARDO VENTURA DIAZ,NaN,19400,942311613,CHIMBOTE,2026-07-11,17:30:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,47651671,FREDY PABLO MILLA ROSALES,NaN,14000,951060809,HUARAZ,2026-07-11,09:15:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,32926030,CARLOS EDGARDO VENTURA DIAZ,942311613,734272 - CHIMBOTE,2026-07-11,19400,Derivacion
1,BOT,TARGET,47651671,FREDY PABLO MILLA ROSALES,951060809,734285 - PC HUARAZ,2026-07-11,14000,Derivacion


In [9]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

1051

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,09750804,AMADO VILLALTA,NaN,23000,942707381,CASTILLA,2026-07-08,14:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,08931784,LUIS GUILLERMO CHUQUIJAJAS,NaN,23000,950955962,VILLA EL SALVADOR 2,2026-07-08,14:30:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,09750804,AMADO VILLALTA,942707381,737490 - CASTILLA,2026-07-08,23000,Derivacion
1,00000001,TARGET,08931784,LUIS GUILLERMO CHUQUIJAJAS,950955962,732249 - VILLA EL SALVADOR 2,2026-07-08,23000,Derivacion
